In [1]:
!apt update
!apt install zstd



Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,828 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,652 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd6

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time

# Start Ollama server in the background
subprocess.Popen(["ollama", "serve"])

# Wait a few seconds for the server to initialize
time.sleep(5)
print("Ollama is running!")

Ollama is running!


In [4]:
!pip install ollama
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [5]:
import ollama
ollama.pull('nomic-embed-text')

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [7]:
# Pull the model (e.g., Llama 3.2 3B)
!ollama pull llama3

In [8]:
import ollama
import chromadb

# --- 1. SAMPLE DATA: Indian Food & Cuisine Context ---
# These act as your "Knowledge Base" for food-related queries
food_data = [
    "Butter Chicken: A popular Indian dish made with chicken cooked in a creamy tomato sauce. Best served with Naan.",
    "Biryani: A fragrant rice dish made with basmati rice, spices, and meat (chicken, mutton) or vegetables. Originates from the Indian subcontinent.",
    "Dosa: A thin, savory crepe from South India, made from a fermented batter of rice and black lentils. Often served with sambar and chutney.",
    "Vada Pav: A vegetarian fast food dish native to the Indian state of Maharashtra. It consists of a deep-fried potato dumpling (vada) placed inside a bread bun (pav) sliced almost in half through the middle.",
    "Indian Spices: Common spices include turmeric, cumin, coriander, garam masala, cardamom, and chili powder.",
    "Jalebi: A sweet and crunchy Indian dessert made by deep-frying flour batter in a pretzel or circular shape, then soaking it in sugar syrup. Often served warm."
]

# --- 2. STORAGE SETUP (Food-Explorer-DB) ---
client = chromadb.PersistentClient(path="./food_explorer_db")
collection = client.get_or_create_collection(name="indian_food_guide")

# --- 3. EMBED & INDEX ---
print("Syncing Food-Explorer with culinary records...")
for i, text in enumerate(food_data):
    response = ollama.embeddings(model="nomic-embed-text", prompt=text)
    collection.add(
        ids=[f"food_item_{i}"],
        embeddings=[response["embedding"]],
        documents=[text]
    )

# --- 4. THE FOOD QUERY ---
user_food_query = "Tell me about a popular Maharashtrian snack and a famous South Indian dish."

# Retrieve relevant snippets
query_emb = ollama.embeddings(model="nomic-embed-text", prompt=user_food_query)["embedding"]
results = collection.query(query_embeddings=[query_emb], n_results=2)
context = "\n".join(results['documents'][0])

# --- 5. GENERATE RESPONSE ---
output = ollama.generate(
    model="llama3",
    system="You are an Indian Food Expert. Provide delicious and informative descriptions.",
    prompt=f"Food Context: {context}\n\nUser Query: {user_food_query}"
)

print("\n--- FOOD EXPLORER INSIGHTS ---")
print(output['response'])


Syncing Food-Explorer with culinary records...

--- FOOD EXPLORER INSIGHTS ---
My friend, I'm delighted to introduce you to two iconic Indian treats that are sure to tantalize your taste buds!

**Maharashtrian Snack: Misal Pav**

Misal Pav is a beloved snack from the state of Maharashtra, particularly from the city of Pune. It's a flavorful and filling treat that's perfect for any time of day. Imagine a bowl of spicy, flavorful curry made with sprouted lentils (moong) and topped with crunchy farsan (savory granola), onions, and coriander leaves. Serve it all over a soft, buttery pav (bread), and you have a match made in heaven! The combination of textures and flavors is simply divine. Misal Pav is often served as a snack or light meal, and its popularity has spread far beyond Maharashtra's borders.

**South Indian Dish: Idli**

Idli is a staple dish from South India, particularly from the states of Tamil Nadu, Karnataka, and Kerala. These soft, fluffy steamed cakes are made from a ferm